# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

Prioritize content for review when it has been a long time since its last update and there is meaningful search demand.

The baseline score increases when a page is older since its last update and when its topic has higher search volume. The score is used to rank pages for human review; it does not automatically mean that a page needs to be changed.

### Reason code

**STALE_HIGH_DEMAND** — The page has both relatively high `days_since_last_update` and relatively high `search_volume`, making it a reasonable candidate for content-refresh review.


In [3]:
import pandas as pd

# Load the dataset
url = "https://github.com/syedamominapak-coder/flyrankai_ml_internship/raw/refs/heads/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset loaded:", df.shape)

# Signal check 1: staleness
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

staleness_table = (
    df["staleness_bucket"]
    .value_counts()
    .sort_index()
    .rename_axis("staleness_bucket")
    .reset_index(name="n")
)

print("\nStaleness signal check:")
print(staleness_table)

Dataset loaded: (30000, 44)

Staleness signal check:
  staleness_bucket      n
0        0-30 days  20480
1       31-90 days    175
2      91-180 days   9171
3        181+ days    174


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import pandas as pd

# Signal check 2: search demand
# Bucket pages by their search volume.

volume_bins = [-1, 10, 50, 100, 500, float("inf")]
volume_labels = [
    "0-10",
    "11-50",
    "51-100",
    "101-500",
    "501+"
]

df["volume_bucket"] = pd.cut(
    df["search_volume"],
    bins=volume_bins,
    labels=volume_labels
)

volume_table = (
    df["volume_bucket"]
    .value_counts()
    .sort_index()
    .rename_axis("volume_bucket")
    .reset_index(name="n")
)

print("Search-volume signal check:")
print(volume_table)

Search-volume signal check:
  volume_bucket      n
0          0-10  18392
1         11-50   4989
2        51-100   1102
3       101-500   2021
4          501+   1028


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
import pandas as pd
import os

# Make sure the dataset is available
if "df" not in globals():
    url = "https://github.com/syedamominapak-coder/flyrankai_ml_internship/raw/refs/heads/main/data/raw/content_refresh_anonymized.csv"
    df = pd.read_csv(url)

# Create percentile-based scores so the two signals are comparable
df["staleness_score"] = df["days_since_last_update"].rank(pct=True)
df["demand_score"] = df["search_volume"].rank(pct=True)

# ONE baseline score
df["baseline_score"] = (
    0.5 * df["staleness_score"] +
    0.5 * df["demand_score"]
)

# One reason code and one action label
df["reason_code"] = "STALE_HIGH_DEMAND"
df["action"] = "REVIEW_FOR_REFRESH"

# Rank highest-priority pages first
queue = (
    df[
        [
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "search_volume"
        ]
    ]
    .sort_values("baseline_score", ascending=False)
    .reset_index(drop=True)
)

queue["rank"] = queue.index + 1

print("Ranked queue created:", queue.shape)
print("\nTop 10:")
print(queue.head(10))

Ranked queue created: (30000, 7)

Top 10:
             content_id  baseline_score        reason_code  \
0  content_a31e10779c01        0.992520  STALE_HIGH_DEMAND   
1  content_bbca724138f2        0.991291  STALE_HIGH_DEMAND   
2  content_40e140ba2934        0.984434  STALE_HIGH_DEMAND   
3  content_24abafed9707        0.978968  STALE_HIGH_DEMAND   
4  content_23e958c54c78        0.975976  STALE_HIGH_DEMAND   
5  content_c3dd69918c8c        0.969697  STALE_HIGH_DEMAND   
6  content_29ec1008c834        0.969697  STALE_HIGH_DEMAND   
7  content_6efb8fa48ebe        0.961461  STALE_HIGH_DEMAND   
8  content_0cc405838fc5        0.961002  STALE_HIGH_DEMAND   
9  content_17e6b2ba4b08        0.956235  STALE_HIGH_DEMAND   

               action  days_since_last_update  search_volume  rank  
0  REVIEW_FOR_REFRESH                     144         3600.0     1  
1  REVIEW_FOR_REFRESH                     236         1600.0     2  
2  REVIEW_FOR_REFRESH                     231          720.0     3  

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# 4. Weak picks + leakage check

# Show the lowest-confidence / potentially weak top-10 picks.
# A weak pick is one where the two signals disagree:
# high demand but very recent update, or very stale but very low demand.

top20 = queue.head(20).copy()

top20["weak_pick"] = (
    (
        (top20["days_since_last_update"] <= df["days_since_last_update"].median()) &
        (top20["search_volume"] >= df["search_volume"].median())
    )
    |
    (
        (top20["days_since_last_update"] >= df["days_since_last_update"].median()) &
        (top20["search_volume"] <= df["search_volume"].median())
    )
)

print("Potential weak picks in top 20:")
print(
    top20[
        [
            "rank",
            "content_id",
            "baseline_score",
            "days_since_last_update",
            "search_volume",
            "reason_code",
            "action",
            "weak_pick"
        ]
    ]
)

# Leakage check
used_features = [
    "days_since_last_update",
    "search_volume"
]

label_or_future_fields = [
    "trend_direction",
    "trend_pct",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d"
]

leaked_fields = [col for col in used_features if col in label_or_future_fields]

print("\nLeakage check:")
print("Features used:", used_features)
print("Label/future fields used:", leaked_fields)

if len(leaked_fields) == 0:
    print("PASS — no label-derived or future-window fields were used.")
else:
    print("FAIL — leakage detected:", leaked_fields)

Potential weak picks in top 20:
    rank            content_id  baseline_score  days_since_last_update  \
0      1  content_a31e10779c01        0.992520                     144   
1      2  content_bbca724138f2        0.991291                     236   
2      3  content_40e140ba2934        0.984434                     231   
3      4  content_24abafed9707        0.978968                     231   
4      5  content_23e958c54c78        0.975976                     144   
5      6  content_c3dd69918c8c        0.969697                     151   
6      7  content_29ec1008c834        0.969697                     151   
7      8  content_6efb8fa48ebe        0.961461                     151   
8      9  content_0cc405838fc5        0.961002                     144   
9     10  content_17e6b2ba4b08        0.956235                     144   
10    11  content_638236e8066e        0.950605                     144   
11    12  content_02b0d6e30129        0.947739                     313   
12    

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.